# Clase 169 — TF Lite / LiteRT (mobile y embedded)

Convertir un modelo a **TensorFlow Lite** (renombrado **LiteRT** en 2024) para móviles, IoT y
embebidos. Aplicar **quantization** (int8) para reducir el tamaño ~4× y acelerar 2-4× en CPU
móvil, e inferir con `tf.lite.Interpreter`.

Requiere: `tensorflow` (opcional). El código de conversión es correcto pero no se ejecuta aquí.

## 1. Convertir un SavedModel a `.tflite`

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    TF_OK = True
except Exception:
    TF_OK = False
    print("tensorflow no instalado -> se muestra la API de conversion (no se ejecuta)")

if TF_OK:
    converter = tf.lite.TFLiteConverter.from_saved_model("servable/1/")
    tflite_model = converter.convert()
    open("model.tflite", "wb").write(tflite_model)
    print(f"model.tflite: {len(tflite_model)} bytes (float32)")
else:
    print("converter = tf.lite.TFLiteConverter.from_saved_model('servable/1/')")
    print("tflite_model = converter.convert()")

## 2. Quantization de rango dinámico

`Optimize.DEFAULT` cuantiza los **pesos** a int8 (activaciones en float en runtime). Reduce el
tamaño ~4× sin dataset de calibración.

In [ ]:
if TF_OK:
    converter = tf.lite.TFLiteConverter.from_saved_model("servable/1/")
    converter.optimizations = [tf.lite.Optimize.DEFAULT]     # dynamic range (pesos int8)
    tflite_dyn = converter.convert()
    print(f"dynamic range: {len(tflite_dyn)} bytes (~4x mas chico)")
else:
    print("converter.optimizations = [tf.lite.Optimize.DEFAULT]")

## 3. Quantization int8 completa (con representative dataset)

Cuantiza **pesos y activaciones** a int8. Necesita un `representative_dataset` (~100 ejemplos)
para calibrar los rangos de las activaciones. Es la que mejor acelera en hardware int8.

In [ ]:
import numpy as np
np.random.seed(42)

def representative_dataset():
    for _ in range(100):
        yield [np.random.rand(1, 784).astype("float32")]     # ~100 ejemplos diversos

if TF_OK:
    converter = tf.lite.TFLiteConverter.from_saved_model("servable/1/")
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    tflite_int8 = converter.convert()
    print(f"int8 full: {len(tflite_int8)} bytes")
else:
    print("converter.representative_dataset = representative_dataset")
    print("converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]")

## 4. Inferencia con `tf.lite.Interpreter`

In [ ]:
if TF_OK:
    interpreter = tf.lite.Interpreter(model_path="model.tflite")
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    x = np.random.rand(*inp["shape"]).astype(inp["dtype"])
    interpreter.set_tensor(inp["index"], x)
    interpreter.invoke()
    y = interpreter.get_tensor(out["index"])
    print("salida Interpreter shape:", y.shape)
else:
    print("interp = tf.lite.Interpreter(model_path='model.tflite'); interp.allocate_tensors()")
    print("interp.set_tensor(idx, x); interp.invoke(); interp.get_tensor(out_idx)")

## 5. Trade-offs y delegates

| Variante | Tamaño | Velocidad | Accuracy |
|---|---|---|---|
| float32 | 1× | 1× | referencia |
| dynamic range | ~1/4 | 1.5-2× | ~igual |
| int8 full | ~1/4 | 2-4× (HW int8) | −0 a 1 pp |

Para acelerar de verdad en el dispositivo se usan **delegates**: NNAPI (Android), CoreML (iOS,
Neural Engine), GPU delegate. Para microcontroladores existe **TF Lite Micro** (modelos < 100 KB).

## 6. Tabla comparativa de las variantes

In [ ]:
import numpy as np
variantes = ["float32", "dynamic_range", "int8_full"]
# tamaños relativos y speedup tipicos (float32 como referencia)
factor_tam = np.array([1.00, 0.27, 0.26])
speedup    = np.array([1.0, 1.8, 3.2])
acc_drop   = np.array([0.0, 0.1, 0.6])          # puntos porcentuales
print(f"{'variante':16s} {'tam_rel':>8s} {'speedup':>8s} {'acc_drop_pp':>12s}")
for v, t, s, a in zip(variantes, factor_tam, speedup, acc_drop):
    print(f"{v:16s} {t:8.2f} {s:8.1f}x {a:11.1f}")
print("-> int8 full: ~4x mas chico, 2-4x mas rapido, <1pp de perdida")

## Ejercicios

1. Convertir un modelo Fashion-MNIST a `.tflite` y comparar el tamaño con el SavedModel.
2. Generar las 3 variantes (float32, dynamic range, int8 full) y tabular tamaño, accuracy y latencia.
3. Verificar que int8 reduce el tamaño ~4× con < 1 pp de pérdida de accuracy.
4. Medir la latencia de inferencia float32 vs int8 en CPU con `Interpreter`.

## Conclusiones

- LiteRT (ex TF Lite) es el runtime para edge: móvil, IoT y embebidos.
- `TFLiteConverter.from_saved_model` convierte; `Interpreter` ejecuta la inferencia.
- `Optimize.DEFAULT` da quantization de rango dinámico; int8 full requiere representative dataset.
- La quantization reduce tamaño ~4× y acelera 2-4×; los delegates explotan el hardware del dispositivo.